#### **CORRECTITUD DE DATOS**

In [781]:
import pandas as pd 
import numpy as np

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")

Creamos una función que nos permita verificar la correctitud de los datos. La idea es calcular manualmente las filas de una columna específica y luego compararla con las de la existente.

In [782]:
def verificar_correctitud(df, col1, col2):
    df = df.copy()
    df['coincide'] = np.isclose(df[col1], df[col2])
    
    # Filtrar las que no coinciden
    no_coinciden = df[df['coincide'] == False]
    
    if no_coinciden.empty:
        print("TODAS LAS FILAS COINCIDEN.")
    else:
        print(f"{len(no_coinciden)}/{len(df)} FILAS NO COINCIDEN.")
        return no_coinciden[[col1, col2, 'coincide']]

1. Chequeo de la dimensión externa de cada tipo de caja.

In [783]:
dimensiones = ['largo', 'ancho', 'alto']

# Calculamos la dimensión externa para cada tipo de caja
for dim in dimensiones:
    especificaciones_cajas[f'caja_exterior_{dim}_calculado'] = (
        especificaciones_cajas[f'caja_interior_{dim}'] + 
        2 * especificaciones_cajas['caja_grosor_mm']
    )

# Verificar
for dim in dimensiones:
    verificar_correctitud(especificaciones_cajas, 
                          f'caja_exterior_{dim}', 
                          f'caja_exterior_{dim}_calculado')
    print("-" * 30)

34/204 FILAS NO COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.
------------------------------


Veamos en detalle las filas no coincidentes en cuanto a la dimensión externa del largo.

In [784]:
verificar_correctitud(especificaciones_cajas, 'caja_exterior_largo', 'caja_exterior_largo_calculado')

34/204 FILAS NO COINCIDEN.


,caja_exterior_largo,caja_exterior_largo_calculado,coincide
0,400.0,400.4,False
11,400.0,400.4,False
17,400.0,400.4,False
21,400.0,400.4,False
25,400.0,400.4,False
27,400.0,400.4,False
33,400.0,400.4,False
34,400.0,400.4,False
36,400.0,400.4,False
44,400.0,400.4,False


Notemos que la diferencia es despreciable, de un 0.4. Trataremos esto como un error de registro.

2. Siguiendo la restricción de column stacking, chequeamos si la cantidad de cajas en altura se corresponde con la relación entre el alto de pallet y el alto de la dimensión externa de caja. Además, chequeamos que sigue una orientación fija, es decir, que el lado largo de la caja sea paralelo al lado ancho del pallet.

In [785]:
especificaciones_cajas['cantidad_cajas_alto_calculado'] = (
    especificaciones_cajas['pallet_alto'] // especificaciones_cajas['caja_exterior_alto']
)

especificaciones_cajas['cantidad_cajas_ancho_calculado'] = (
    especificaciones_cajas['pallet_largo'] // especificaciones_cajas['caja_exterior_ancho']
)

especificaciones_cajas['cantidad_cajas_largo_calculado'] = (
    especificaciones_cajas['pallet_ancho'] // especificaciones_cajas['caja_exterior_largo']
)

verificar_correctitud(especificaciones_cajas, 'cantidad_cajas_alto', 
                                'cantidad_cajas_alto_calculado')
print("-" * 30)
verificar_correctitud(especificaciones_cajas, 'cantidad_cajas_ancho', 
                                'cantidad_cajas_ancho_calculado')
print("-" * 30)
verificar_correctitud(especificaciones_cajas, 'cantidad_cajas_largo', 
                                'cantidad_cajas_largo_calculado')

TODAS LAS FILAS COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.


3. Chequeamos el cálculo de utilización de pallet.

In [786]:
especificaciones_cajas['utilizacion_calculado'] = (
    especificaciones_cajas['caja_exterior_alto'] * 
    especificaciones_cajas['caja_exterior_ancho'] *
    especificaciones_cajas['caja_exterior_largo'] *
    especificaciones_cajas['cantidad_cajas_total'] / (1800 * 800 * 1200)
)

verificar_correctitud(especificaciones_cajas, 'utilizacion', 
                                'utilizacion_calculado')

TODAS LAS FILAS COINCIDEN.


4. Chequeo del volumen total de cada producto.

In [787]:
operaciones_planta['volumen_producto_total_calculado'] = (
    operaciones_planta['volumen_producto_canal_servicios_comida'] +
    operaciones_planta['volumen_producto_canal_minorista'] +
    operaciones_planta['volumen_producto_canal_cadenas_corporativas'] +
    operaciones_planta['volumen_producto_canal_otros']
)

verificar_correctitud(operaciones_planta, 'volumen_producto_total', 
                                'volumen_producto_total_calculado')

TODAS LAS FILAS COINCIDEN.


In [788]:
operaciones_planta['volumen_producto_total_calculado'] = (
    operaciones_planta['volumen_producto_planta_buenos_aires'] +
    operaciones_planta['volumen_producto_planta_curitiba'] +
    operaciones_planta['volumen_producto_planta_santiago'] +
    operaciones_planta['volumen_producto_planta_monterrey'] +
    operaciones_planta['volumen_producto_planta_bakersfield']
)

verificar_correctitud(operaciones_planta, 'volumen_producto_total', 
                                'volumen_producto_total_calculado')

TODAS LAS FILAS COINCIDEN.


5. Chequeo de la cantidad y costo total de pallets.

In [789]:
operaciones_planta['cantidad_pallets_total_calculado'] = (
    operaciones_planta['cantidad_pallets_planta_buenos_aires'] +
    operaciones_planta['cantidad_pallets_planta_curitiba'] +
    operaciones_planta['cantidad_pallets_planta_santiago'] +
    operaciones_planta['cantidad_pallets_planta_monterrey'] +
    operaciones_planta['cantidad_pallets_planta_bakersfield']
)

verificar_correctitud(operaciones_planta, 'cantidad_pallets_total',
                                'cantidad_pallets_total_calculado')

TODAS LAS FILAS COINCIDEN.


In [790]:
operaciones_planta['costo_pallets_total_calculado'] = (
    operaciones_planta['costo_pallets_planta_buenos_aires'] +
    operaciones_planta['costo_pallets_planta_curitiba'] +
    operaciones_planta['costo_pallets_planta_santiago'] +
    operaciones_planta['costo_pallets_planta_monterrey'] +
    operaciones_planta['costo_pallets_planta_bakersfield']
)

verificar_correctitud(operaciones_planta, 'costo_pallets_total',
                                'costo_pallets_total_calculado')

TODAS LAS FILAS COINCIDEN.


In [791]:
operaciones_planta['costo_total_calculado'] = (
    operaciones_planta['costo_total_planta_buenos_aires'] +
    operaciones_planta['costo_total_planta_curitiba'] +
    operaciones_planta['costo_total_planta_santiago'] +
    operaciones_planta['costo_total_planta_monterrey'] +
    operaciones_planta['costo_total_planta_bakersfield']
)

verificar_correctitud(operaciones_planta, 'costo_total',
                                'costo_total_calculado')

TODAS LAS FILAS COINCIDEN.


6. Chequeamos que el costo de pallets por planta sea coherente teniendo en cuenta que cada costo de flete es de inicialmente 150 USD

In [792]:
# Lista de plantas
plantas = ['buenos_aires', 'curitiba', 'santiago', 'monterrey', 'bakersfield']

# Calculamos los costos de pallets para cada planta
for planta in plantas:
    cantidad = f'cantidad_pallets_planta_{planta}'
    costo_calc = f'costo_pallets_planta_{planta}_calculado'
    operaciones_planta[costo_calc] = 150 * operaciones_planta[cantidad]

# Verificar
for planta in plantas:
    verificar_correctitud(operaciones_planta, 
                          f'costo_pallets_planta_{planta}', 
                          f'costo_pallets_planta_{planta}_calculado')
    print("-" * 30)

TODAS LAS FILAS COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.
------------------------------


7. Chequeemos que para cada producto, el volumen total para cada planta equivale a n_pallets * cajas_por_pallet.

In [793]:
prod_op_merge = catalogo_productos.merge(operaciones_planta, on="codigo_producto")

prod_caja_op_merge = prod_op_merge.merge(especificaciones_cajas,
                                         on='caja_tipo_id', how='left')

# Lista de plantas
plantas = ['buenos_aires', 'curitiba', 'santiago', 'monterrey', 'bakersfield']

# Calculamos el volumen en base a n_pallets para cada planta
for planta in plantas:
    cant_pallets = f'cantidad_pallets_planta_{planta}'
    volumen_calc = f'volumen_producto_planta_{planta}_calculado'
    prod_caja_op_merge[volumen_calc] = prod_caja_op_merge[cant_pallets] * prod_caja_op_merge['cantidad_cajas_total']

# Verificar
for planta in plantas:
    verificar_correctitud(prod_caja_op_merge, 
                          f'volumen_producto_planta_{planta}', 
                          f'volumen_producto_planta_{planta}_calculado')
    print("-" * 30)

128/427 FILAS NO COINCIDEN.
------------------------------
203/427 FILAS NO COINCIDEN.
------------------------------
169/427 FILAS NO COINCIDEN.
------------------------------
85/427 FILAS NO COINCIDEN.
------------------------------
97/427 FILAS NO COINCIDEN.
------------------------------


Notemos que algunas filas no coinciden, sobre todo las de la planta de Curitiba. Pensamos que esto puede deberse a que algún pallet no esté totalmente completo, por lo que compararemos esa diferencia por la cantidad de cajas que entran en un pallet.

In [794]:
for planta in plantas:
    volumen_original = f'volumen_producto_planta_{planta}'
    volumen_calculado = f'volumen_producto_planta_{planta}_calculado'
    # Sumamos lo que calculamos a la cantidad de cajas dentro del pallet
    cumplen = prod_caja_op_merge[volumen_original] <= prod_caja_op_merge[volumen_calculado] + prod_caja_op_merge['cantidad_cajas_total']
    if len(cumplen) == len(catalogo_productos):
        print("TODAS LAS FILAS COINCIDEN.")
    else:
        print(f"{len(catalogo_productos) - len(cumplen)}/{len(catalogo_productos)} FILAS NO COINCIDEN.")
    print("-" * 30)

TODAS LAS FILAS COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.
------------------------------


8. Chequeamos que para cada planta, el volumen de tipos de cajas en el archivo de compras es mayor o igual al volumen total de productos que usan ese tipo de caja determinado. Si sucediese lo contrario, habrían productos sin envase secundario, lo cual no tendría sentido.

In [795]:
plantas = ['buenos_aires', 'curitiba', 'santiago', 'monterrey', 'bakersfield']

for planta in plantas:
    volumen_compras = f'volumen_tipo_planta_{planta}'
    volumen_requerido = f'volumen_tipo_planta_{planta}_requerido'
    cumplen = procurement_cajas[volumen_requerido] <= procurement_cajas[volumen_compras]
    if len(cumplen) == len(procurement_cajas):
        print("TODAS LAS FILAS COINCIDEN.")
    else:
        print(f"{len(procurement_cajas) - len(cumplen)}/{len(procurement_cajas)} FILAS NO COINCIDEN.")
    print("-" * 30)

TODAS LAS FILAS COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.
------------------------------


9. Para cada producto, chequeamos los cálculos de costo de packaging según la planta correspondiente, aplicando el respectivo descuento.

In [796]:
prod_op_merge = catalogo_productos.merge(operaciones_planta, on="codigo_producto")

prod_caja_op_merge = prod_op_merge.merge(especificaciones_cajas,
                                         on='caja_tipo_id', how='left')

prod_caja_op_procure_merge = prod_caja_op_merge.merge(procurement_cajas,
                                                      on='caja_tipo_id')

# Lista de plantas
plantas = ['buenos_aires', 'curitiba', 'santiago', 'monterrey', 'bakersfield']

# Calculamos los costos de packaging para cada planta
for planta in plantas:
    volumen = f'volumen_producto_planta_{planta}'
    costo_uni = f'costo_unitario_planta_{planta}'
    costo_calc = f'costo_total_planta_{planta}_calculado'
    prod_caja_op_procure_merge[costo_calc] = prod_caja_op_procure_merge[volumen] * prod_caja_op_procure_merge[costo_uni]

# Verificar
for planta in plantas:
    verificar_correctitud(prod_caja_op_procure_merge, 
                          f'costo_total_planta_{planta}', 
                          f'costo_total_planta_{planta}_calculado')
    print("-" * 30)

TODAS LAS FILAS COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.
------------------------------
9/427 FILAS NO COINCIDEN.
------------------------------
TODAS LAS FILAS COINCIDEN.
------------------------------
4/427 FILAS NO COINCIDEN.
------------------------------


10. Chequeamos si el costo total base está bien calculado.

In [797]:
# El costo total está calculado en base a las cajas utilizadas para cada producto
procurement_cajas['costo_total_base_calculado'] = (
    (procurement_cajas['volumen_tipo_planta_buenos_aires_requerido'] +
     procurement_cajas['volumen_tipo_planta_curitiba_requerido'] +
     procurement_cajas['volumen_tipo_planta_santiago_requerido'] +
     procurement_cajas['volumen_tipo_planta_monterrey_requerido'] +
     procurement_cajas['volumen_tipo_planta_bakersfield_requerido']) * procurement_cajas['costo_unitario_base']
)

verificar_correctitud(procurement_cajas, 'costo_total_base',
                                        'costo_total_base_calculado')

TODAS LAS FILAS COINCIDEN.
